In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder \
                    .appName("Processing Sports Items Data with RDDs, DataFrames, and Spark SQL") \
                    .master("local[*]") \
                    .getOrCreate()

### Using RDD

In [5]:
sport_items_rdd = spark.sparkContext.textFile("../Data/Sports Items.csv")

In [6]:
sport_items_rdd

SportItems.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [7]:
sport_items_rdd.collect()

['Football,18',
 'cricket bat ,25',
 'Badminton racket,10',
 'Tennis ball,23',
 'Football,11',
 'cricket bat ,22',
 'Badminton racket,13',
 'Tennis ball,33',
 'Football,17',
 'cricket bat ,45',
 'Badminton racket,11',
 'Tennis ball,22',
 'Football,98',
 'cricket bat ,33',
 'Badminton racket,11',
 'Tennis ball,55',
 'Football,71',
 'cricket bat ,56',
 'Badminton racket,91',
 'Tennis ball,89']

In [8]:
pairs = sport_items_rdd.map(lambda row: row.split(',')) \
                        .map(lambda fields: (fields[0], int(fields[1])))

In [9]:
pairs.collect()

[('Football', 18),
 ('cricket bat ', 25),
 ('Badminton racket', 10),
 ('Tennis ball', 23),
 ('Football', 11),
 ('cricket bat ', 22),
 ('Badminton racket', 13),
 ('Tennis ball', 33),
 ('Football', 17),
 ('cricket bat ', 45),
 ('Badminton racket', 11),
 ('Tennis ball', 22),
 ('Football', 98),
 ('cricket bat ', 33),
 ('Badminton racket', 11),
 ('Tennis ball', 55),
 ('Football', 71),
 ('cricket bat ', 56),
 ('Badminton racket', 91),
 ('Tennis ball', 89)]

In [10]:
pairs.reduceByKey(lambda x, y: x + y).collect()

[('cricket bat ', 181),
 ('Tennis ball', 222),
 ('Football', 215),
 ('Badminton racket', 136)]

### Using DataFrames

In [11]:
sports_schema = StructType([
    StructField("Item", StringType(), nullable=True),
    StructField("Quantity", IntegerType(), nullable=False)
])

In [12]:
sport_items = spark.read \
                    .format('csv') \
                    .option('header', 'false') \
                    .option('inferSchema', 'false') \
                    .schema(sports_schema) \
                    .load('../Data/Sports Items.csv')

In [13]:
sport_items.describe()

DataFrame[summary: string, Item: string, Quantity: string]

In [14]:
sport_items.show()

+----------------+--------+
|            Item|Quantity|
+----------------+--------+
|        Football|      18|
|    cricket bat |      25|
|Badminton racket|      10|
|     Tennis ball|      23|
|        Football|      11|
|    cricket bat |      22|
|Badminton racket|      13|
|     Tennis ball|      33|
|        Football|      17|
|    cricket bat |      45|
|Badminton racket|      11|
|     Tennis ball|      22|
|        Football|      98|
|    cricket bat |      33|
|Badminton racket|      11|
|     Tennis ball|      55|
|        Football|      71|
|    cricket bat |      56|
|Badminton racket|      91|
|     Tennis ball|      89|
+----------------+--------+



#### Using Spark SQL

In [15]:
sport_items_view = sport_items.createOrReplaceTempView("sport_items")

In [16]:
spark.sql('''
            select Item, 
                    sum(Quantity) as total_quantity
            from sport_items 
            group by Item
            ''').collect()

[Row(Item='Tennis ball', total_quantity=222),
 Row(Item='Badminton racket', total_quantity=136),
 Row(Item='cricket bat ', total_quantity=181),
 Row(Item='Football', total_quantity=215)]

#### Using the DataFrame API

In [17]:
sport_items.groupBy("Item") \
                  .agg(F.sum("Quantity").alias("total_quantity")) \
                  .orderBy(F.desc("total_quantity")) \
                  .show()

+----------------+--------------+
|            Item|total_quantity|
+----------------+--------------+
|     Tennis ball|           222|
|        Football|           215|
|    cricket bat |           181|
|Badminton racket|           136|
+----------------+--------------+



In [18]:
sport_items.groupBy("Item") \
  .agg(
    F.sum("Quantity").alias("total_quantity"),
    F.count("*").alias("occurrences")
  ) \
  .orderBy("Item") \
  .show()

+----------------+--------------+-----------+
|            Item|total_quantity|occurrences|
+----------------+--------------+-----------+
|Badminton racket|           136|          5|
|        Football|           215|          5|
|     Tennis ball|           222|          5|
|    cricket bat |           181|          5|
+----------------+--------------+-----------+



In [19]:
spark.stop()